In [1]:
# %% Cell 1 - imports, CUDA check, paths

from pathlib import Path
import importlib
import sys
from datetime import datetime

import h5py
import numpy as np
import torch


def log(message: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "This notebook kernel does not have CUDA. Select the CUDA-enabled Python environment "
        "before training; CPU fallback is intentionally disabled."
    )
log(f"Training on CUDA: {torch.cuda.get_device_name(0)}")

root = Path.cwd().resolve()
while root != root.parent and not (root / "clean_simulation").is_dir():
    root = root.parent
if not (root / "clean_simulation").is_dir():
    raise RuntimeError("Could not find workspace root containing clean_simulation/")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

DATA_GROUP = "integration_2_poly_1"
data_dir = root / "clean_simulation" / "outputs" / "data"
checkpoint_dir = root / "clean_simulation" / "outputs" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
training_h5 = data_dir / "grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30_clean.h5"
CHECKPOINT_PREFIX = "grid_locked_source_sdf_rebuilt_queso"

if not training_h5.is_file():
    raise FileNotFoundError(
        f"Missing generated dataset: {training_h5}. Run the initial-SDF / rebuilt-QuESo generation cell in 01_data_generation.ipynb first."
    )
print(f"Using training HDF5: {training_h5.name}")

[23:58:29] Training on CUDA: Quadro T1000
Using training HDF5: grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30_clean.h5


In [2]:
# %% Full-cell complement reference verification
# The complement inverse K_full - K is valid only if K_full is in the same
# normalized framework convention as the HDF5 targets: K_framework / (E*h).

fem_kernel_dir = root / "clean_simulation" / "internal_fem_kernel"
if str(fem_kernel_dir) not in sys.path:
    sys.path.insert(0, str(fem_kernel_dir))

from constants import Material
from integration.element_stiffness import Stiffness
import clean_simulation.internal_nn_runtime.cutcell_ml.model as cutcell_model_module

importlib.reload(cutcell_model_module)
HEX_CORNER_COORDS = cutcell_model_module.HEX_CORNER_COORDS
K_FULL_E1_F64 = cutcell_model_module.K_FULL_E1_F64
self_check_K_full = cutcell_model_module.self_check_K_full

reference_h = 0.01
framework_full = Stiffness.elements(
    HEX_CORNER_COORDS.to(torch.float64) * (reference_h / 2.0),
    Material(E=1.0, nu=0.3).C,
    order=2,
    device="cpu",
    poly_order=1,
).cpu() / reference_h
full_reference_rel_error = float(
    torch.linalg.matrix_norm(framework_full - K_FULL_E1_F64, ord="fro")
    / torch.linalg.matrix_norm(framework_full, ord="fro")
)
print(f"framework normalized K_full relative error: {full_reference_rel_error:.3e}")
assert full_reference_rel_error < 1.0e-12
self_check_K_full(K_FULL_E1_F64)

framework normalized K_full relative error: 3.395e-16
symmetry max abs error:      1.388e-17  (expect ~1e-14)
min eigenvalue:              -9.834625e-17  (expect ~0 from rigid modes)
rigid-body K@R max abs err:  8.327e-17  (expect ~1e-12 or smaller)


In [12]:
# %% Cell 2 - verify the aligned initial-SDF / rebuilt-QuESo dataset

with h5py.File(training_h5, "r") as training_file:
    training_group = training_file[DATA_GROUP]

    required = {
        "flat_index",
        "d_values",
        "stiffness_ut",
        "cut_volume_fraction",
        "geometry_name",
    }
    missing = required.difference(training_group.keys())
    if missing:
        raise KeyError(f"Generated dataset is missing required fields: {sorted(missing)}")

    geometry_names = training_group["geometry_name"].asstr()[...]

    SIMPLE_GEOMETRIES = [
        "V7_text",
        "lvl0_binary",
        "hanging_sphere_surface",
    ]
    HARSH_GEOMETRIES = [
        "eggcarton_sphere",
        "thin_ring_sphere",
        "star_lobe_sphere",
        "spike_ball",
    ]
    CRYSTAL_GEOMETRIES = [f"crystal_{index:04d}" for index in range(30)]
    wanted = set(SIMPLE_GEOMETRIES + HARSH_GEOMETRIES + CRYSTAL_GEOMETRIES)

    mask = np.isin(geometry_names, list(wanted))
    indices = np.where(mask)[0]
    print(f"Selected {len(indices):,} / {len(geometry_names):,} samples")
    print("Matched geometries:", np.unique(geometry_names[indices]))

    d_values = training_group["d_values"][indices]
    stiffness_ut = training_group["stiffness_ut"][indices]
    cut_volume_fraction = training_group["cut_volume_fraction"][indices]
    geometry_names = geometry_names[indices]

    available = len(indices)

    if d_values.shape != (available, 8):
        raise ValueError(
            f"Unexpected d_values shape: {d_values.shape}, expected {(available, 8)}"
        )

    if stiffness_ut.shape != (available, 300):
        raise ValueError(
            f"Unexpected stiffness_ut shape: {stiffness_ut.shape}, expected {(available, 300)}"
        )

    if not np.isfinite(d_values).all() or not np.isfinite(stiffness_ut).all():
        raise ValueError("Generated dataset contains non-finite training values")

    if training_group.attrs.get("input_source") != "initial_stl_signed_distance_aligned_to_queso_material_side":
        raise ValueError(
            "Dataset inputs are not orientation-aligned initial-STL signed distances"
        )

    if training_group.attrs.get("target_source") != "rebuilt_grid_locked_marching_cubes_queso":
        raise ValueError(
            "Dataset targets are not QuESo stiffness from the rebuilt geometry"
        )

    if training_group.attrs.get("cell_selection") != "all_rebuilt_queso_trimmed_cells":
        raise ValueError(
            "Dataset does not retain every rebuilt QuESo cut cell"
        )

    if training_group.attrs.get("selection_filter") != "none":
        raise ValueError("Dataset has an unexpected row-selection filter")

print(
    f"Aligned initial-SDF / rebuilt-QuESo dataset verified: {available:,} rows; "
    f"volume fraction range [{cut_volume_fraction.min():.6f}, "
    f"{cut_volume_fraction.max():.6f}]"
)

Selected 1,149,504 / 1,149,504 samples
Matched geometries: ['V7_text' 'crystal_0000' 'crystal_0001' 'crystal_0002' 'crystal_0003'
 'crystal_0004' 'crystal_0005' 'crystal_0006' 'crystal_0007'
 'crystal_0008' 'crystal_0009' 'crystal_0010' 'crystal_0011'
 'crystal_0012' 'crystal_0013' 'crystal_0014' 'crystal_0015'
 'crystal_0016' 'crystal_0017' 'crystal_0018' 'crystal_0019'
 'crystal_0020' 'crystal_0021' 'crystal_0022' 'crystal_0023'
 'crystal_0024' 'crystal_0025' 'crystal_0026' 'crystal_0027'
 'crystal_0028' 'crystal_0029' 'eggcarton_sphere' 'hanging_sphere_surface'
 'lvl0_binary' 'spike_ball' 'star_lobe_sphere' 'thin_ring_sphere']
Aligned initial-SDF / rebuilt-QuESo dataset verified: 1,149,504 rows; volume fraction range [0.001000, 1.000000]


In [4]:
# %% Cell 3 - direct training view
# The generator already provides every rebuilt QuESo cut cell with initial-SDF
# features. Do not apply source/rebuild agreement, classification, or occupancy filters.

print(f"Training directly on all {available:,} rebuilt QuESo cut cells.")

Training directly on all 224,585 rebuilt QuESo cut cells.


In [5]:
# %% Cell 4 - finalize

log(
    f"Ready: {available:,} initial-SDF -> rebuilt-QuESo stiffness rows; "
    f"dataset={training_group.name}"
)

[17:35:00] Ready: 224,585 initial-SDF -> rebuilt-QuESo stiffness rows; dataset=None


In [2]:
# %% Cell 5 — imports (unchanged)

import clean_simulation.internal_nn_runtime.cutcell_ml.config as cutcell_cfg_module
import clean_simulation.internal_nn_runtime.workflows as workflows_module
importlib.reload(cutcell_cfg_module)
importlib.reload(workflows_module)

DataConfig = cutcell_cfg_module.DataConfig
RunConfig = cutcell_cfg_module.RunConfig
TrainingConfig = cutcell_cfg_module.TrainingConfig
from clean_simulation.internal_nn_runtime.workflows import run_training_pipeline

In [5]:
# %% Learnability preflight
# This uses the same physically constrained architecture as the held-out probe.

from clean_simulation.internal_nn_runtime.cutcell_ml.data.datasets import reconstruct_matrices_from_ut
from clean_simulation.internal_nn_runtime.cutcell_ml.model import CutCellNet
from clean_simulation.internal_nn_runtime.cutcell_ml.classification_utils import get_class_label_batch
from clean_simulation.internal_nn_runtime.cutcell_ml.training import composite_loss

PREFLIGHT_SAMPLES = 200
PREFLIGHT_STEPS = 800
PREFLIGHT_FUSION_DIM = 1024
PREFLIGHT_DEPTH = 4
PREFLIGHT_SEED = 15
device = torch.device("cuda")

if available < PREFLIGHT_SAMPLES:
    raise ValueError(f"Need at least {PREFLIGHT_SAMPLES} rows for the learnability preflight")
preflight_indices = np.sort(
    np.random.default_rng(PREFLIGHT_SEED).choice(available, size=PREFLIGHT_SAMPLES, replace=False)
)
with h5py.File(training_h5, "r") as training_file:
    training_group = training_file[DATA_GROUP]
    d_preflight = torch.as_tensor(training_group["d_values"][preflight_indices], dtype=torch.float32, device=device)
    k_preflight = torch.as_tensor(
        reconstruct_matrices_from_ut(training_group["stiffness_ut"][preflight_indices]).astype(np.float32),
        device=device,
    )
    print(training_group["geometry_name"])

model_preflight = CutCellNet(
    fusion_dim=PREFLIGHT_FUSION_DIM,
    n_residual_blocks=PREFLIGHT_DEPTH,
    trunk_type="resnet",
    residual_scaling="rezero",
    spd_mode="cholesky",
    use_reduced_null_space=True,
    matrix_size=24,
    n_distance_nodes=8,
).to(device)
model_preflight.set_normalizers(
    d_preflight.mean(dim=0).cpu(),
    d_preflight.std(dim=0).clamp_min(1.0e-4).cpu(),
)
optimizer_preflight = torch.optim.AdamW(model_preflight.parameters(), lr=2.0e-3, weight_decay=0.0)
nu_preflight = torch.full((PREFLIGHT_SAMPLES,), 0.3, device=device)
class_preflight = get_class_label_batch(d_preflight)


def preflight_loss() -> torch.Tensor:
    return composite_loss(
        model_preflight, d_preflight, nu_preflight, k_preflight,
        cut_class=class_preflight,
    )[0]

with torch.no_grad():
    initial_preflight_loss = float(preflight_loss())
for _ in range(PREFLIGHT_STEPS):
    optimizer_preflight.zero_grad(set_to_none=True)
    loss = preflight_loss()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_preflight.parameters(), max_norm=1.0)
    optimizer_preflight.step()
with torch.no_grad():
    final_preflight_loss, preflight_metrics = composite_loss(
        model_preflight, d_preflight, nu_preflight, k_preflight,
        cut_class=class_preflight,
    )
preflight_ratio = float(final_preflight_loss / initial_preflight_loss)
preflight_rel_l2 = float(preflight_metrics["rel_l2"])
log(
    f"{PREFLIGHT_SAMPLES}-row random mostly-inside preflight: {initial_preflight_loss:.5f} -> "
    f"{float(final_preflight_loss):.5f}; ratio={preflight_ratio:.3f}; rel L2={preflight_rel_l2:.3f}"
)


<HDF5 dataset "geometry_name": shape (1149504,), type "|S64">
[20:24:36] 200-row random mostly-inside preflight: 14.53020 -> 0.18486; ratio=0.013; rel L2=0.185


In [8]:
training_group.items()

ItemsViewHDF5(<Closed HDF5 group>)

In [5]:
probe_checkpoint = checkpoint_dir / "source_sdf_rebuilt_queso_d6_f2048.pt"

In [6]:
probe_checkpoint_2 = checkpoint_dir / "source_sdf_rebuilt_queso_d4_f256_2.pt"

In [6]:
def save_checkpoint(model, history, epoch, val_loss):
    torch.save(
        {
            "epoch": epoch,
            "val_loss": val_loss,
            "model_state_dict": model.state_dict(),
            "history": history,
        },
        probe_checkpoint,
    )

In [13]:
# %% Materialize expanded all-crystal training data
filtered_h5 = data_dir / "grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30_filtered.h5"

with h5py.File(training_h5, "r") as fin, h5py.File(filtered_h5, "w") as fout:
    in_group = fin[DATA_GROUP]
    out_group = fout.create_group(DATA_GROUP)

    geometry_names = in_group["geometry_name"].asstr()[...]
    indices = np.where(np.isin(geometry_names, list(wanted)))[0]

    for name, obj in in_group.items():
        if isinstance(obj, h5py.Dataset):
            out_group.create_dataset(name, data=obj[indices])

    for key, value in in_group.attrs.items():
        out_group.attrs[key] = value

log(f"Materialized {len(indices):,} expanded rows in {filtered_h5.name}")

[11:26:50] Materialized 1,149,504 expanded rows in grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30_filtered.h5


In [ ]:
# %% Expanded-data transfer calibration
EXPANDED_CALIBRATION_EPOCHS = 3
EXPANDED_CALIBRATION_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_all_crystals_calibration.pt"

expanded_calibration_config = make_probe_config(
    depth=10,
    epochs=EXPANDED_CALIBRATION_EPOCHS,
    lr=1.0e-5,
)
expanded_calibration_config.training.scheduler_type = "none"
expanded_calibration_config.training.early_stopping_patience = EXPANDED_CALIBRATION_EPOCHS + 1
expanded_calibration_config.training.early_stopping_min_delta = 0.0

torch.cuda.empty_cache()
expanded_calibration = run_training_pipeline(
    config=expanded_calibration_config,
    base_dir=root,
    init_checkpoint_path=FRIDAY_LONG_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(EXPANDED_CALIBRATION_CHECKPOINT),
)
expanded_calibration_best = min(expanded_calibration.history.val_loss)
log(
    f"Expanded-data calibration complete: best val rel L2={expanded_calibration_best:.6%}; "
    f"checkpoint={EXPANDED_CALIBRATION_CHECKPOINT.name}"
)

Dataset group has 1,149,504 rows; loading all rows.
Loaded HDF5 rows: 200,000/1,149,504 (17.4%)
Loaded HDF5 rows: 1,000,000/1,149,504 (87.0%)
Loaded HDF5 rows: 1,149,504/1,149,504 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_rebuilt_queso_d10_f1024_8h_resume.pt
Beginning training over 3 epochs (loss: relative_frobenius)
Epoch    1/   3 | Train: 0.006670 | Val: 0.006591 | Val rel_l2: 0.66% | Val ref_norm_p50: 1.003 | Lr: 1.50e-06 | Time: 1452.70s
Epoch    2/   3 | Train: 0.006601 | Val: 0.006580 | Val rel_l2: 0.66% | Val ref_norm_p50: 1.003 | Lr: 1.50e-06 | Time: 1421.03s
Epoch    3/   3 | Train: 0.006563 | Val: 0.006553 | Val rel_l2: 0.66% | Val ref_norm_p50: 1.003 | Lr: 1.50e-06 | Time: 1430.99s

--- Final Evaluation ---
  Val loss                 : 6.552696e-03
  Val rel_l2                : 0.655270%
  Symmetry error (K-K^T)   : 8.196e-08  (expect ~0, sanity check)
  Rigid-body

In [17]:
# %% Batch-size fit check for expanded training
from clean_simulation.internal_nn_runtime.cutcell_ml.data.datasets import reconstruct_matrices_from_ut
from clean_simulation.internal_nn_runtime.cutcell_ml.classification_utils import get_class_label_batch
from clean_simulation.internal_nn_runtime.cutcell_ml.model import CutCellNet
from clean_simulation.internal_nn_runtime.cutcell_ml.training import composite_loss

if "expanded_calibration" in globals():
    del expanded_calibration
torch.cuda.empty_cache()

BATCH_SIZE_CANDIDATE = 512
with h5py.File(filtered_h5, "r") as training_file:
    batch_d = torch.as_tensor(
        training_file[DATA_GROUP]["d_values"][:BATCH_SIZE_CANDIDATE],
        dtype=torch.float32,
        device="cuda",
    )
    batch_k = torch.as_tensor(
        reconstruct_matrices_from_ut(
            training_file[DATA_GROUP]["stiffness_ut"][:BATCH_SIZE_CANDIDATE]
        ).astype(np.float32),
        device="cuda",
    )

batch_nu = torch.full((BATCH_SIZE_CANDIDATE,), 0.3, device="cuda")
batch_cls = get_class_label_batch(batch_d)
batch_model = CutCellNet(
    fusion_dim=1024,
    n_residual_blocks=10,
    trunk_type="resnet",
    residual_scaling="rezero",
    spd_mode="cholesky",
    use_reduced_null_space=True,
    matrix_size=24,
    n_distance_nodes=8,
).cuda()
batch_model.load_state_dict(
    torch.load(EXPANDED_CALIBRATION_CHECKPOINT, map_location="cuda", weights_only=False)["model_state_dict"]
)
batch_optimizer = torch.optim.AdamW(batch_model.parameters(), lr=1.5e-6)

batch_optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast("cuda"):
    batch_loss = composite_loss(batch_model, batch_d, batch_nu, batch_k, cut_class=batch_cls)[0]
batch_loss.backward()
batch_optimizer.step()
torch.cuda.synchronize()
log(f"Batch-size {BATCH_SIZE_CANDIDATE} fits: loss={float(batch_loss):.6f}")

del batch_optimizer, batch_model, batch_d, batch_k, batch_nu, batch_cls
torch.cuda.empty_cache()

[12:54:00] Batch-size 512 fits: loss=0.004626


C:\Users\z0058p1u\AppData\Local\Temp\ipykernel_32160\464162239.py:48: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  log(f"Batch-size {BATCH_SIZE_CANDIDATE} fits: loss={float(batch_loss):.6f}")


In [18]:
# %% GPU calibration for fresh and Friday-resume models
CALIBRATION_EPOCHS = 3
CALIBRATION_BATCH_SIZE = 512
FRIDAY_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d4_f256_2.pt"
FRESH_CALIBRATION_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d16_f1024_calibration.pt"
FRIDAY_CALIBRATION_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_calibration.pt"

if not FRIDAY_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Friday checkpoint is missing: {FRIDAY_CHECKPOINT}")


def make_probe_config(*, depth: int, epochs: int, lr: float) -> RunConfig:
    return RunConfig(
        data=DataConfig(
            h5_path=filtered_h5,
            depth=2,
            integration_order=2,
            poly_order=1,
            poisson_ratio=0.3,
            min_samples=0,
            max_load_samples=None,
            refined_subcells=1,
            value_resolution=1,
        ),
        training=TrainingConfig(
            model_type="cutcellnet",
            batch_size=CALIBRATION_BATCH_SIZE,
            fusion_dim=1024,
            n_residual_blocks=depth,
            trunk_type="resnet",
            residual_scaling="rezero",
            spd_mode="cholesky",
            use_reduced_null_space=True,
            matrix_size=24,
            dropout_p=0.0,
            weight_decay=1.0e-5,
            grad_clip_norm=1.0,
            epochs=epochs,
            lr=lr,
            scheduler_type="none",
            train_split=0.9,
            seed=15,
            standardize_inputs=True,
            use_decoupled_loss=False,
            physical_loss_weight=1.0,
            classifier_weight=0.0,
            fourier_loss_weight=0.0,
            energy_loss_weight=0.0,
            nullspace_loss_weight=0.0,
            soft_mode_loss_weight=0.0,
            warmup_phys_only_epochs=0,
            aux_ramp_epochs=0,
            early_stopping_patience=CALIBRATION_EPOCHS + 1,
            early_stopping_min_delta=0.0,
            num_workers=0,
            pin_memory=True,
            persistent_workers=False,
            use_amp=True,
        ),
    )


def checkpoint_saver(path: Path):
    def save(model, history, epoch, val_loss):
        torch.save(
            {
                "epoch": epoch,
                "val_loss": val_loss,
                "model_state_dict": model.state_dict(),
                "history": history,
            },
            path,
        )
    return save


def summarize_calibration(label: str, summary) -> dict[str, float]:
    history = summary.history
    result = {
        "initial_val_loss": float(history.val_loss[0]),
        "final_val_loss": float(history.val_loss[-1]),
        "relative_change": float(history.val_loss[-1] / history.val_loss[0] - 1.0),
    }
    log(
        f"{label} calibration: val {result['initial_val_loss']:.6f} -> "
        f"{result['final_val_loss']:.6f} ({result['relative_change']:+.2%})"
    )
    return result

# These jobs are sequential because their combined GPU footprint exceeds 4 GiB.
torch.cuda.empty_cache()
fresh_calibration = run_training_pipeline(
    config=make_probe_config(depth=16, epochs=CALIBRATION_EPOCHS, lr=7.0e-5),
    base_dir=root,
    init_checkpoint_path=None,
    checkpoint_callback=checkpoint_saver(FRESH_CALIBRATION_CHECKPOINT),
)
fresh_calibration_metrics = summarize_calibration("Fresh d16/f1024", fresh_calibration)
del fresh_calibration
torch.cuda.empty_cache()

friday_calibration = run_training_pipeline(
    config=make_probe_config(depth=10, epochs=CALIBRATION_EPOCHS, lr=1.0e-5),
    base_dir=root,
    init_checkpoint_path=FRIDAY_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(FRIDAY_CALIBRATION_CHECKPOINT),
)
friday_calibration_metrics = summarize_calibration("Friday d10/f1024", friday_calibration)
del friday_calibration
torch.cuda.empty_cache()

Dataset group has 1,149,504 rows; loading all rows.
Loaded HDF5 rows: 200,000/1,149,504 (17.4%)
Loaded HDF5 rows: 1,000,000/1,149,504 (87.0%)
Loaded HDF5 rows: 1,149,504/1,149,504 (100.0%)
Executing training on device: cuda
Beginning training over 3 epochs (loss: relative_frobenius)
Epoch    1/   3 | Train: 0.206200 | Val: 0.069684 | Val rel_l2: 6.97% | Val ref_norm_p50: 1.003 | Lr: 7.00e-05 | Time: 1308.29s
Epoch    2/   3 | Train: 0.068381 | Val: 0.059949 | Val rel_l2: 5.99% | Val ref_norm_p50: 1.003 | Lr: 7.00e-05 | Time: 1535.67s
Epoch    3/   3 | Train: 0.060953 | Val: 0.055070 | Val rel_l2: 5.51% | Val ref_norm_p50: 1.003 | Lr: 7.00e-05 | Time: 1642.33s

--- Final Evaluation ---
  Val loss                 : 5.506959e-02
  Val rel_l2                : 5.506959%
  Symmetry error (K-K^T)   : 8.941e-08  (expect ~0, sanity check)
  Rigid-body residual (KR) : 2.831e-07  (expect ~0)
  Min physical eigenvalue  : 2.059046e-11  (should be > 0)
[14:09:41] Fresh d16/f1024 calibration: val 0.0

In [ ]:
# %% Sequential depth-and-width architecture calibration
# Every candidate starts from scratch and runs alone because the GPU has 4 GiB VRAM.
from itertools import product

ARCHITECTURE_CALIBRATION_EPOCHS = 1000
ARCHITECTURE_CALIBRATION_LR = 1.0e-5
ARCHITECTURE_DEPTHS = (4, 8, 12, 16)
ARCHITECTURE_WIDTHS = (256, 512, 1024, 2048)
ARCHITECTURE_SWEEP_PATH = checkpoint_dir / "source_sdf_rebuilt_queso_architecture_sweep.csv"


def make_architecture_probe_config(*, depth: int, width: int) -> RunConfig:
    config = make_probe_config(
        depth=depth,
        epochs=ARCHITECTURE_CALIBRATION_EPOCHS,
        lr=ARCHITECTURE_CALIBRATION_LR,
    )
    config.training.fusion_dim = width
    config.training.scheduler_type = "cosine"
    config.training.early_stopping_patience = ARCHITECTURE_CALIBRATION_EPOCHS + 1
    config.training.early_stopping_min_delta = 0.0
    return config


sweep_rows = []
for depth, width in product(ARCHITECTURE_DEPTHS, ARCHITECTURE_WIDTHS):
    candidate_checkpoint = checkpoint_dir / f"source_sdf_rebuilt_queso_d{depth}_f{width}_architecture_calibration.pt"
    log(f"Starting fresh architecture calibration: depth={depth}, width={width}")

    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started_at = datetime.now()

    calibration = run_training_pipeline(
        config=make_architecture_probe_config(depth=depth, width=width),
        base_dir=root,
        init_checkpoint_path=None,
        checkpoint_callback=checkpoint_saver(candidate_checkpoint),
    )
    validation_losses = np.asarray(calibration.history.val_loss, dtype=np.float64)
    best_epoch = int(np.argmin(validation_losses)) + 1
    best_val_loss = float(validation_losses[best_epoch - 1])
    final_val_loss = float(validation_losses[-1])
    parameter_count = int(sum(parameter.numel() for parameter in calibration.model.parameters()))
    elapsed_seconds = (datetime.now() - started_at).total_seconds()

    sweep_rows.append({
        "depth": depth,
        "fusion_dim": width,
        "parameter_count": parameter_count,
        "initial_val_loss": float(validation_losses[0]),
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "final_val_loss": final_val_loss,
        "elapsed_seconds": elapsed_seconds,
        "checkpoint": str(candidate_checkpoint),
    })
    architecture_sweep = pd.DataFrame(sweep_rows).sort_values(
        ["best_val_loss", "final_val_loss", "parameter_count"]
    ).reset_index(drop=True)
    architecture_sweep.to_csv(ARCHITECTURE_SWEEP_PATH, index=False)
    log(
        f"d{depth}/f{width}: best val={best_val_loss:.6f} at epoch {best_epoch}; "
        f"final={final_val_loss:.6f}; elapsed={elapsed_seconds / 60.0:.1f} min"
    )

    del calibration
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

best_architecture = architecture_sweep.iloc[0]
log(
    f"Best architecture: d{int(best_architecture['depth'])}/"
    f"f{int(best_architecture['fusion_dim'])}, minimum validation loss="
    f"{best_architecture['best_val_loss']:.6f} at epoch "
    f"{int(best_architecture['best_epoch'])}."
)
log(f"Saved architecture ranking to {ARCHITECTURE_SWEEP_PATH}")
display(architecture_sweep)

In [9]:
# %% Eight-hour sequential continuation
# The Quadro T1000 has 4 GiB VRAM, so the two models cannot run concurrently.
# Measured at batch size 384, 31 fresh epochs plus 49 Friday-resume epochs
# takes approximately eight GPU-hours including data reload and final checks.
FRESH_LONG_EPOCHS = 31
FRIDAY_LONG_EPOCHS = 49
FRESH_LONG_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d16_f1024_8h.pt"
FRIDAY_LONG_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_8h_resume.pt"

long_run_started = datetime.now()
log("Starting sequential eight-hour continuation: fresh d16/f1024, then Friday d10/f1024.")

torch.cuda.empty_cache()
fresh_long_run = run_training_pipeline(
    config=make_probe_config(depth=16, epochs=FRESH_LONG_EPOCHS, lr=7.0e-5),
    base_dir=root,
    init_checkpoint_path=FRESH_CALIBRATION_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(FRESH_LONG_CHECKPOINT),
)
fresh_long_best = min(fresh_long_run.history.val_loss)
log(f"Fresh continuation complete: best val loss={fresh_long_best:.6f}; checkpoint={FRESH_LONG_CHECKPOINT.name}")
del fresh_long_run
torch.cuda.empty_cache()

# The three-epoch calibration drifted slightly at 1e-5, so lower the
# continuation rate while retaining a small cosine decay for stability.
friday_long_config = make_probe_config(depth=10, epochs=FRIDAY_LONG_EPOCHS, lr=3.0e-6)
friday_long_config.training.scheduler_type = "cosine"
friday_long_config.training.cosine_eta_min_factor = 0.5
friday_long_config.training.early_stopping_patience = 16
friday_long_config.training.early_stopping_min_delta = 1.0e-6
friday_long_run = run_training_pipeline(
    config=friday_long_config,
    base_dir=root,
    init_checkpoint_path=FRIDAY_CALIBRATION_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(FRIDAY_LONG_CHECKPOINT),
)
friday_long_best = min(friday_long_run.history.val_loss)
log(f"Friday continuation complete: best val loss={friday_long_best:.6f}; checkpoint={FRIDAY_LONG_CHECKPOINT.name}")
log(f"Total continuation wall time: {datetime.now() - long_run_started}")

[01:24:49] Starting sequential eight-hour continuation: fresh d16/f1024, then Friday d10/f1024.
Dataset group has 536,339 rows; loading all rows.
Loaded HDF5 rows: 200,000/536,339 (37.3%)
Loaded HDF5 rows: 536,339/536,339 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_rebuilt_queso_d16_f1024_calibration.pt
Beginning training over 31 epochs (loss: relative_frobenius)
Epoch    1/  31 | Train: 0.067990 | Val: 0.070932 | Val rel_l2: 7.09% | Val ref_norm_p50: 1.011 | Lr: 7.00e-05 | Time: 461.16s
Epoch    2/  31 | Train: 0.061147 | Val: 0.060780 | Val rel_l2: 6.08% | Val ref_norm_p50: 1.011 | Lr: 7.00e-05 | Time: 462.94s
Epoch    3/  31 | Train: 0.057047 | Val: 0.081641 | Val rel_l2: 8.16% | Val ref_norm_p50: 1.011 | Lr: 7.00e-05 | Time: 491.29s
Epoch    4/  31 | Train: 0.055498 | Val: 0.054911 | Val rel_l2: 5.49% | Val ref_norm_p50: 1.011 | Lr: 7.00e-05 | Time: 476.72s
Epoch    5/  31 |

In [10]:
# %% Remaining-time fresh refinement
# Earlier early stopping ended the planned jobs before the eight-hour budget.
# Continue from the best fresh checkpoint at half the successful calibration LR.
FRESH_REFINEMENT_EPOCHS = 16
FRESH_REFINEMENT_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d16_f1024_8h_refined.pt"

fresh_refinement_config = make_probe_config(
    depth=16,
    epochs=FRESH_REFINEMENT_EPOCHS,
    lr=3.5e-5,
)
fresh_refinement_config.training.scheduler_type = "cosine"
fresh_refinement_config.training.cosine_eta_min_factor = 0.5
fresh_refinement_config.training.early_stopping_patience = FRESH_REFINEMENT_EPOCHS + 1
fresh_refinement_config.training.early_stopping_min_delta = 0.0

torch.cuda.empty_cache()
fresh_refinement = run_training_pipeline(
    config=fresh_refinement_config,
    base_dir=root,
    init_checkpoint_path=FRESH_LONG_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(FRESH_REFINEMENT_CHECKPOINT),
)
fresh_refinement_best = min(fresh_refinement.history.val_loss)
log(
    f"Fresh refinement complete: best val loss={fresh_refinement_best:.6f}; "
    f"checkpoint={FRESH_REFINEMENT_CHECKPOINT.name}"
)

Dataset group has 536,339 rows; loading all rows.
Loaded HDF5 rows: 200,000/536,339 (37.3%)
Loaded HDF5 rows: 536,339/536,339 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_rebuilt_queso_d16_f1024_8h.pt
Beginning training over 16 epochs (loss: relative_frobenius)
Epoch    1/  16 | Train: 0.033039 | Val: 0.039581 | Val rel_l2: 3.96% | Val ref_norm_p50: 1.011 | Lr: 3.48e-05 | Time: 462.41s
Epoch    2/  16 | Train: 0.031443 | Val: 0.029804 | Val rel_l2: 2.98% | Val ref_norm_p50: 1.011 | Lr: 3.43e-05 | Time: 460.69s
Epoch    3/  16 | Train: 0.031482 | Val: 0.032565 | Val rel_l2: 3.26% | Val ref_norm_p50: 1.011 | Lr: 3.35e-05 | Time: 472.57s
Epoch    4/  16 | Train: 0.030675 | Val: 0.032911 | Val rel_l2: 3.29% | Val ref_norm_p50: 1.011 | Lr: 3.24e-05 | Time: 461.12s
Epoch    5/  16 | Train: 0.029763 | Val: 0.028849 | Val rel_l2: 2.88% | Val ref_norm_p50: 1.011 | Lr: 3.11e-05 | Time: 473

In [19]:
# %% Ten-hour target run: Friday-derived model
# Continue from the all-crystal calibration so validation covers the expanded distribution.
# Batch size 512 passed the real forward/backward fit check; 33 epochs use approximately 10 hours.
TARGET_REL_L2 = 0.001
TARGET_RUN_EPOCHS = 33
TARGET_RUN_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_all_crystals_10h.pt"
TARGET_RUN_STARTED = datetime.now()

target_config = make_probe_config(
    depth=10,
    epochs=TARGET_RUN_EPOCHS,
    lr=1.5e-6,
)
target_config.training.scheduler_type = "cosine"
target_config.training.cosine_eta_min_factor = 0.25
target_config.training.early_stopping_patience = TARGET_RUN_EPOCHS + 1
target_config.training.early_stopping_min_delta = 0.0

torch.cuda.empty_cache()
target_run = run_training_pipeline(
    config=target_config,
    base_dir=root,
    init_checkpoint_path=FRIDAY_CALIBRATION_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(TARGET_RUN_CHECKPOINT),
)
target_best = min(target_run.history.val_loss)
log(
    f"Target run complete: best val rel L2={target_best:.6%}; "
    f"target met={target_best <= TARGET_REL_L2}; checkpoint={TARGET_RUN_CHECKPOINT.name}; "
    f"wall time={datetime.now() - TARGET_RUN_STARTED}"
)

Dataset group has 1,149,504 rows; loading all rows.
Loaded HDF5 rows: 200,000/1,149,504 (17.4%)
Loaded HDF5 rows: 1,000,000/1,149,504 (87.0%)
Loaded HDF5 rows: 1,149,504/1,149,504 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_rebuilt_queso_d10_f1024_calibration.pt
Beginning training over 33 epochs (loss: relative_frobenius)
Epoch    1/  33 | Train: 0.006299 | Val: 0.006517 | Val rel_l2: 0.65% | Val ref_norm_p50: 1.003 | Lr: 1.50e-06 | Time: 1062.58s
Epoch    2/  33 | Train: 0.006274 | Val: 0.006504 | Val rel_l2: 0.65% | Val ref_norm_p50: 1.003 | Lr: 1.49e-06 | Time: 1098.12s
Epoch    3/  33 | Train: 0.006251 | Val: 0.006488 | Val rel_l2: 0.65% | Val ref_norm_p50: 1.003 | Lr: 1.48e-06 | Time: 1075.50s
Epoch    4/  33 | Train: 0.006237 | Val: 0.006482 | Val rel_l2: 0.65% | Val ref_norm_p50: 1.003 | Lr: 1.46e-06 | Time: 1043.08s
Epoch    5/  33 | Train: 0.006219 | Val: 0.006473 | Val

In [ ]:
# %% Learning-rate restart check from the completed all-crystal checkpoint
LR_RESTART_EPOCHS = 2
LR_RESTART = 5.0e-6
LR_RESTART_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_all_crystals_lr5e6_calibration.pt"

if "target_run" in globals():
    del target_run
torch.cuda.empty_cache()

lr_restart_config = make_probe_config(
    depth=10,
    epochs=LR_RESTART_EPOCHS,
    lr=LR_RESTART,
)
lr_restart_config.training.scheduler_type = "none"
lr_restart_config.training.early_stopping_patience = LR_RESTART_EPOCHS + 1
lr_restart_config.training.early_stopping_min_delta = 0.0

lr_restart_run = run_training_pipeline(
    config=lr_restart_config,
    base_dir=root,
    init_checkpoint_path=TARGET_RUN_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(LR_RESTART_CHECKPOINT),
)
lr_restart_best = min(lr_restart_run.history.val_loss)
log(
    f"LR restart check: best val rel L2={lr_restart_best:.6%}; "
    f"checkpoint={LR_RESTART_CHECKPOINT.name}"
)
del lr_restart_run
torch.cuda.empty_cache()

Dataset group has 1,149,504 rows; loading all rows.
Loaded HDF5 rows: 200,000/1,149,504 (17.4%)
Loaded HDF5 rows: 1,000,000/1,149,504 (87.0%)
Loaded HDF5 rows: 1,149,504/1,149,504 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_rebuilt_queso_d10_f1024_all_crystals_10h.pt
Beginning training over 2 epochs (loss: relative_frobenius)
Epoch    1/   2 | Train: 0.006050 | Val: 0.006394 | Val rel_l2: 0.64% | Val ref_norm_p50: 1.003 | Lr: 5.00e-06 | Time: 1444.92s
Epoch    2/   2 | Train: 0.006017 | Val: 0.006382 | Val rel_l2: 0.64% | Val ref_norm_p50: 1.003 | Lr: 5.00e-06 | Time: 951.90s

--- Final Evaluation ---
  Val loss                 : 6.382316e-03
  Val rel_l2                : 0.638232%
  Symmetry error (K-K^T)   : 7.451e-08  (expect ~0, sanity check)
  Rigid-body residual (KR) : 2.906e-07  (expect ~0)
  Min physical eigenvalue  : 9.153355e-11  (should be > 0)
[22:44:13] LR restart c

In [22]:
# %% Free GPU memory before the higher-rate continuation
import gc

for name in (
    "batch_loss", "framework_full", "K_FULL_E1_F64", "HEX_CORNER_COORDS",
    "fresh_refinement", "friday_long_run", "preflight_metrics",
    "d_preflight", "k_preflight", "nu_preflight", "class_preflight",
    "model_preflight", "optimizer_preflight",
 ):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(
    f"CUDA allocated={torch.cuda.memory_allocated() / 2**20:.0f} MiB; "
    f"reserved={torch.cuda.memory_reserved() / 2**20:.0f} MiB"
 )

CUDA allocated=3131 MiB; reserved=3300 MiB


In [23]:
# %% Release the interrupted training traceback and its CUDA tensors
import gc
import sys

for name in (
    "last_traceback", "last_type", "last_value",
 ):
    if hasattr(sys, name):
        delattr(sys, name)

ipython = get_ipython()
for name in ("_last_traceback", "_last_execution_result"):
    if hasattr(ipython, name):
        setattr(ipython, name, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(
    f"CUDA allocated={torch.cuda.memory_allocated() / 2**20:.0f} MiB; "
    f"reserved={torch.cuda.memory_reserved() / 2**20:.0f} MiB"
 )

CUDA allocated=3131 MiB; reserved=3300 MiB


In [3]:
# %% Continuation-only setup after a kernel restart
filtered_h5 = data_dir / "grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30_filtered.h5"
LR_RESTART_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_all_crystals_lr5e6_calibration.pt"
TARGET_REL_L2 = 0.001

if not filtered_h5.is_file():
    raise FileNotFoundError(f"Missing expanded training data: {filtered_h5}")
if not LR_RESTART_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Missing validated restart checkpoint: {LR_RESTART_CHECKPOINT}")

def make_probe_config(*, depth: int, epochs: int, lr: float) -> RunConfig:
    return RunConfig(
        data=DataConfig(
            h5_path=filtered_h5,
            depth=2,
            integration_order=2,
            poly_order=1,
            poisson_ratio=0.3,
            min_samples=0,
            max_load_samples=None,
            refined_subcells=1,
            value_resolution=1,
        ),
        training=TrainingConfig(
            model_type="cutcellnet",
            batch_size=512,
            fusion_dim=1024,
            n_residual_blocks=depth,
            trunk_type="resnet",
            residual_scaling="rezero",
            spd_mode="cholesky",
            use_reduced_null_space=True,
            matrix_size=24,
            dropout_p=0.0,
            weight_decay=1.0e-5,
            grad_clip_norm=1.0,
            epochs=epochs,
            lr=lr,
            scheduler_type="none",
            train_split=0.9,
            seed=15,
            standardize_inputs=True,
            use_decoupled_loss=False,
            physical_loss_weight=1.0,
            classifier_weight=0.0,
            fourier_loss_weight=0.0,
            energy_loss_weight=0.0,
            nullspace_loss_weight=0.0,
            soft_mode_loss_weight=0.0,
            warmup_phys_only_epochs=0,
            aux_ramp_epochs=0,
            early_stopping_patience=epochs + 1,
            early_stopping_min_delta=0.0,
            num_workers=0,
            pin_memory=True,
            persistent_workers=False,
            use_amp=True,
        ),
    )

def checkpoint_saver(path: Path):
    def save(model, history, epoch, val_loss):
        torch.save(
            {
                "epoch": epoch,
                "val_loss": val_loss,
                "model_state_dict": model.state_dict(),
                "history": history,
            },
            path,
        )
    return save

print(f"Continuation setup ready: {filtered_h5.name}; restart={LR_RESTART_CHECKPOINT.name}")

Continuation setup ready: grid_locked_source_sdf_rebuilt_queso_i2_p1_simple_harsh_crystals30_filtered.h5; restart=source_sdf_rebuilt_queso_d10_f1024_all_crystals_lr5e6_calibration.pt


In [4]:
# %% Eight-hour continuation from the validated higher-rate restart
LR_CONTINUATION_EPOCHS = 26
LR_CONTINUATION = 5.0e-6
LR_CONTINUATION_CHECKPOINT = checkpoint_dir / "source_sdf_rebuilt_queso_d10_f1024_all_crystals_lr5e6_8h.pt"
LR_CONTINUATION_STARTED = datetime.now()

torch.cuda.empty_cache()
lr_continuation_config = make_probe_config(
    depth=10,
    epochs=LR_CONTINUATION_EPOCHS,
    lr=LR_CONTINUATION,
)
lr_continuation_config.training.scheduler_type = "cosine"
lr_continuation_config.training.cosine_eta_min_factor = 0.5
lr_continuation_config.training.early_stopping_patience = LR_CONTINUATION_EPOCHS + 1
lr_continuation_config.training.early_stopping_min_delta = 0.0

lr_continuation_run = run_training_pipeline(
    config=lr_continuation_config,
    base_dir=root,
    init_checkpoint_path=LR_RESTART_CHECKPOINT,
    checkpoint_callback=checkpoint_saver(LR_CONTINUATION_CHECKPOINT),
)
lr_continuation_best = min(lr_continuation_run.history.val_loss)
log(
    f"Higher-LR continuation complete: best val rel L2={lr_continuation_best:.6%}; "
    f"target met={lr_continuation_best <= TARGET_REL_L2}; "
    f"checkpoint={LR_CONTINUATION_CHECKPOINT.name}; "
    f"wall time={datetime.now() - LR_CONTINUATION_STARTED}"
)

Dataset group has 1,149,504 rows; loading all rows.
Loaded HDF5 rows: 200,000/1,149,504 (17.4%)
Loaded HDF5 rows: 1,000,000/1,149,504 (87.0%)
Loaded HDF5 rows: 1,149,504/1,149,504 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_rebuilt_queso_d10_f1024_all_crystals_lr5e6_calibration.pt
Beginning training over 26 epochs (loss: relative_frobenius)


C:\Users\z0058p1u\Documents\code\clean_simulation\internal_nn_runtime\workflows.py:356: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if (device == "cuda" and config.training.use_amp) else None
C:\Users\z0058p1u\Documents\code\clean_simulation\internal_nn_runtime\workflows.py:166: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch    1/  26 | Train: 0.005994 | Val: 0.006360 | Val rel_l2: 0.64% | Val ref_norm_p50: 1.003 | Lr: 4.99e-06 | Time: 3366.26s
Epoch    2/  26 | Train: 0.005960 | Val: 0.006373 | Val rel_l2: 0.64% | Val ref_norm_p50: 1.003 | Lr: 4.96e-06 | Time: 3375.36s
Epoch    3/  26 | Train: 0.005935 | Val: 0.006328 | Val rel_l2: 0.63% | Val ref_norm_p50: 1.003 | Lr: 4.92e-06 | Time: 3287.88s
Epoch    4/  26 | Train: 0.005910 | Val: 0.006318 | Val rel_l2: 0.63% | Val ref_norm_p50: 1.003 | Lr: 4.86e-06 | Time: 3385.92s
Epoch    5/  26 | Train: 0.005885 | Val: 0.006319 | Val rel_l2: 0.63% | Val ref_norm_p50: 1.003 | Lr: 4.78e-06 | Time: 3377.41s
Epoch    6/  26 | Train: 0.005859 | Val: 0.006302 | Val rel_l2: 0.63% | Val ref_norm_p50: 1.003 | Lr: 4.69e-06 | Time: 3475.65s
Epoch    7/  26 | Train: 0.005836 | Val: 0.006269 | Val rel_l2: 0.63% | Val ref_norm_p50: 1.003 | Lr: 4.58e-06 | Time: 3387.46s
Epoch    8/  26 | Train: 0.005812 | Val: 0.006267 | Val rel_l2: 0.63% | Val ref_norm_p50: 1.003 | Lr: 4.

KeyboardInterrupt: 

In [8]:
# %% Held-out mostly-material probe
# Run Cells 1-5 and the mostly-material preflight before this cell.

PROBE_EPOCHS = 150
PROBE_DEPTH = 16
PROBE_FUSION_DIM = 1024
PROBE_BATCH_SIZE = 4096
probe_checkpoint = checkpoint_dir / "source_sdf_to_mc_queso_majority_material_d16_f2048_6ep.pt"

probe_config = RunConfig(
    data=DataConfig(
        h5_path=filtered_h5,
        depth=2,
        integration_order=2,
        poly_order=1,
        poisson_ratio=0.3,
        min_samples=0,
        max_load_samples=None,
        refined_subcells=1,
        value_resolution=1,
    ),
    training=TrainingConfig(
        model_type="cutcellnet",
        batch_size=PROBE_BATCH_SIZE,
        fusion_dim=PROBE_FUSION_DIM,
        n_residual_blocks=PROBE_DEPTH,
        trunk_type="resnet",
        residual_scaling="rezero",
        spd_mode="cholesky",
        use_reduced_null_space=True,
        matrix_size=24,
        dropout_p=0.0,
        weight_decay=1.0e-5,
        grad_clip_norm=1.0,
        epochs=PROBE_EPOCHS,
        lr=7.0e-5,
        scheduler_type="cosine",
        cosine_eta_min_factor=0.05,
        train_split=0.9,
        seed=15,
        standardize_inputs=True,
        use_decoupled_loss=False,
        physical_loss_weight=1.0,
        classifier_weight=0.0,
        fourier_loss_weight=0.0,
        energy_loss_weight=0.0,
        nullspace_loss_weight=0.0,
        soft_mode_loss_weight=0.0,
        warmup_phys_only_epochs=0,
        aux_ramp_epochs=0,
        early_stopping_patience=20,
        early_stopping_min_delta=1.0e-4,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False,
        use_amp=False,
    ),
)
probe_summary = run_training_pipeline(
    config=probe_config,
    base_dir=root,
    init_checkpoint_path=None,
    checkpoint_callback=save_checkpoint,
) 
probe_best = min(probe_summary.history.val_loss)
log(f"Mostly-material {PROBE_EPOCHS}-epoch probe complete: best val loss={probe_best:.6f}; checkpoint={probe_checkpoint.name}")

Dataset group has 536,339 rows; loading all rows.
Loaded HDF5 rows: 200,000/536,339 (37.3%)
Loaded HDF5 rows: 536,339/536,339 (100.0%)
Executing training on device: cuda
Beginning training over 150 epochs (loss: relative_frobenius)
Epoch    1/ 150 | Train: 1.300547 | Val: 0.334860 | Val rel_l2: 33.49% | Val ref_norm_p50: 1.011 | Lr: 7.00e-05 | Time: 386.96s


KeyboardInterrupt: 

In [9]:
#second training
# %% Held-out mostly-material probe
# Run Cells 1-5 and the mostly-material preflight before this cell.

PROBE_EPOCHS = 100
PROBE_DEPTH = 10
PROBE_FUSION_DIM = 1024
PROBE_BATCH_SIZE = 256
probe_checkpoint = checkpoint_dir / "source_sdf_to_mc_queso_majority_material_d4_f1024_60ep.pt"

probe_config = RunConfig(
    data=DataConfig(
        h5_path=filtered_h5,
        depth=2,
        integration_order=2,
        poly_order=1,
        poisson_ratio=0.3,
        min_samples=0,
        max_load_samples=None,
        refined_subcells=1,
        value_resolution=1,
    ),
    training=TrainingConfig(
        model_type="cutcellnet",
        batch_size=PROBE_BATCH_SIZE,
        fusion_dim=PROBE_FUSION_DIM,
        n_residual_blocks=PROBE_DEPTH,
        trunk_type="resnet",
        residual_scaling="rezero",
        spd_mode="cholesky",
        use_reduced_null_space=True,
        matrix_size=24,
        dropout_p=0.0,
        weight_decay=1.0e-5,
        grad_clip_norm=1.0,
        epochs=PROBE_EPOCHS,
        lr=1.0e-5,
        scheduler_type="cosine",
        cosine_eta_min_factor=0.5,
        train_split=0.9,
        seed=15,
        standardize_inputs=True,
        use_decoupled_loss=False,
        physical_loss_weight=1.0,
        classifier_weight=0.0,
        fourier_loss_weight=0.0,
        energy_loss_weight=0.0,
        nullspace_loss_weight=0.0,
        soft_mode_loss_weight=0.0,
        warmup_phys_only_epochs=0,
        aux_ramp_epochs=0,
        early_stopping_patience=20,
        early_stopping_min_delta=1.0e-5,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False,
        use_amp=False,
    ),
)
probe_summary = run_training_pipeline(
    config=probe_config,
    base_dir=root,
    init_checkpoint_path=probe_checkpoint,
    checkpoint_callback=save_checkpoint,
) 
probe_best = min(probe_summary.history.val_loss)
log(f"Mostly-material {PROBE_EPOCHS}-epoch probe complete: best val loss={probe_best:.6f}; checkpoint={probe_checkpoint.name}")

Dataset group has 416,324 rows; loading all rows.


Loaded HDF5 rows: 200,000/416,324 (48.0%)
Loaded HDF5 rows: 416,324/416,324 (100.0%)
Executing training on device: cuda
Resuming/initializing from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_to_mc_queso_majority_material_d4_f1024_60ep.pt
Beginning training over 100 epochs (loss: relative_frobenius)
Epoch    1/ 100 | Train: 0.007200 | Val: 0.007252 | Val rel_l2: 0.73% | Val ref_norm_p50: 1.001 | Lr: 1.00e-05 | Time: 266.82s
Epoch    2/ 100 | Train: 0.006846 | Val: 0.007144 | Val rel_l2: 0.71% | Val ref_norm_p50: 1.001 | Lr: 1.00e-05 | Time: 90.39s
Epoch    3/ 100 | Train: 0.006672 | Val: 0.007015 | Val rel_l2: 0.70% | Val ref_norm_p50: 1.001 | Lr: 9.99e-06 | Time: 79.58s
Epoch    4/ 100 | Train: 0.006514 | Val: 0.006889 | Val rel_l2: 0.69% | Val ref_norm_p50: 1.001 | Lr: 9.98e-06 | Time: 78.93s
Epoch    5/ 100 | Train: 0.006386 | Val: 0.006748 | Val rel_l2: 0.67% | Val ref_norm_p50: 1.001 | Lr: 9.97e-06 | Time: 78.62s
Epoch    6/ 100 | Train: 0.0062

In [ ]:
# %% 50-epoch follow-up probe
# Run only after the 10-epoch regression probe passes.

PROBE_EPOCHS = 50
PROBE_DEPTH = 4
PROBE_FUSION_DIM = 512
PROBE_BATCH_SIZE = 1024

probe_config = RunConfig(
    data=DataConfig(
        h5_path=training_h5,
        depth=2,
        integration_order=2,
        poly_order=1,
        poisson_ratio=0.3,
        min_samples=0,
        max_load_samples=None,
        refined_subcells=1,
        value_resolution=1,
    ),
    training=TrainingConfig(
        model_type="cutcellnet",
        batch_size=PROBE_BATCH_SIZE,
        fusion_dim=PROBE_FUSION_DIM,
        n_residual_blocks=PROBE_DEPTH,
        trunk_type="resnet",
        residual_scaling="rezero",
        spd_mode="cholesky",
        use_reduced_null_space=True,
        matrix_size=24,
        dropout_p=0.0,
        weight_decay=1.0e-5,
        grad_clip_norm=1.0,
        epochs=PROBE_EPOCHS,
        lr=2.0e-4,
        scheduler_type="cosine",
        cosine_eta_min_factor=0.5,
        train_split=0.9,
        seed=15,
        standardize_inputs=False,
        use_decoupled_loss=False,
        physical_loss_weight=1.0,
        classifier_weight=0.0,
        fourier_loss_weight=0.0,
        energy_loss_weight=0.0,
        nullspace_loss_weight=0.0,
        soft_mode_loss_weight=0.0,
        warmup_phys_only_epochs=0,
        aux_ramp_epochs=0,
        early_stopping_patience=20,
        early_stopping_min_delta=1.0e-4,
        num_workers=0,
        pin_memory=True,
        persistent_workers=False,
        use_amp=False,
    ),
)
probe_checkpoint = checkpoint_dir / "source_sdf_rebuilt_queso_d4_f512_50ep_fresh.pt"
probe_summary = train_cutcell_model(
    config=probe_config,
    base_dir=root,
    init_checkpoint_path=None,
    checkpoint_path=probe_checkpoint,
)
probe_best = min(probe_summary["val_losses"])
log(f"50-epoch probe complete: best val loss={probe_best:.6f}; checkpoint={probe_checkpoint.name}")

Starting training workflow with base_dir=C:\Users\z0058p1u\Documents\code
Dataset group has 286,520 rows; loading all rows.
Loaded HDF5 rows: 200,000/286,520 (69.8%)
Loaded HDF5 rows: 286,520/286,520 (100.0%)
Executing training on device: cuda
Resuming/Initializing with weights from: C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_to_mc_queso_raw_d4_f512_10ep.pt
Beginning training over 200 epochs
Epoch    1/ 200 | Train Loss: 0.022701 | Train Rel L2: 5.40% | Val Loss: 0.026725 | Val Phys: 0.026725 | Val Rel L2: 6.11% | Lr: 5.00e-05 | Time: 19.11s
Saved best checkpoint at epoch 1 (val_loss=0.026725) to C:\Users\z0058p1u\Documents\code\clean_simulation\outputs\checkpoints\source_sdf_to_mc_queso_raw_d4_f1024_200ep_1.pt
Epoch    2/ 200 | Train Loss: 0.021671 | Train Rel L2: 5.26% | Val Loss: 0.021283 | Val Phys: 0.021283 | Val Rel L2: 5.23% | Lr: 5.00e-05 | Time: 17.79s
Saved best checkpoint at epoch 2 (val_loss=0.021283) to C:\Users\z0058p1u\Documents\code

KeyboardInterrupt: 

In [12]:
import subprocess
import yaml  # PyYAML library

def generate_requirements_yml(output_file="requirements.yml"):
    try:
        # Get installed packages with pip freeze
        result = subprocess.run(
            ["pip", "freeze"],
            capture_output=True,
            text=True,
            check=True,
        )

        # Parse pip freeze output into a dictionary
        packages = {}
        for line in result.stdout.splitlines():
            if "==" in line:
                name, version = line.split("==", 1)
                packages[name.strip()] = version.strip()

        # Save as YAML
        with open(output_file, "w") as f:
            yaml.dump({"dependencies": packages}, f, sort_keys=False)

        print(f"✅ Generated {output_file}")
        return output_file
    except subprocess.CalledProcessError as e:
        print(f"❌ Error: {e.stderr}")
        return None

# Usage
generate_requirements_yml()

✅ Generated requirements.yml


'requirements.yml'